# 03 — Speech Evaluation Metrics: WER and CER

After a TTS or ASR model produces a transcript, how do we measure quality?

This notebook covers the two standard metrics:
1. **WER** — Word Error Rate, the standard ASR benchmark metric
2. **CER** — Character Error Rate, preferred for languages without word boundaries

These metrics apply to both **ASR** (did the model transcribe correctly?) and **TTS** evaluation (run ASR on synthesized speech, then compute WER against the original text).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("Evaluation notebook ready")

In [ ]:
def edit_distance(ref, hyp):
    R, H = len(ref), len(hyp)
    dp = [[0] * (H + 1) for _ in range(R + 1)]
    for i in range(R + 1): dp[i][0] = i
    for j in range(H + 1): dp[0][j] = j
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j],      # deletion
                                    dp[i][j-1],      # insertion
                                    dp[i-1][j-1])    # substitution
    # Traceback
    i, j = R, H
    S = D = I = 0
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            S += 1; i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    return S, D, I, dp[R][H]

def wer(reference, hypothesis):
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    S, D, I, _ = edit_distance(ref, hyp)
    return (S + D + I) / max(len(ref), 1) * 100, S, D, I

def cer(reference, hypothesis):
    ref = list(reference.lower().replace(' ', ''))
    hyp = list(hypothesis.lower().replace(' ', ''))
    S, D, I, _ = edit_distance(ref, hyp)
    return (S + D + I) / max(len(ref), 1) * 100, S, D, I

# Examples
test_cases = [
    ("the cat sat on the mat",  "the cat sat on mat",            "deletion"),
    ("hello world",             "hello word",                    "substitution"),
    ("speech recognition",      "speech recognition system",     "insertion"),
    ("how are you today",       "how are you today",             "perfect"),
    ("recognize speech",        "wreck a nice beach",            "phonetically similar!"),
]

print("WER Examples:")
print(f"  {'Reference':<28} {'Hypothesis':<32} {'WER':>6}  (S/D/I)")
print("  " + "-" * 80)
for ref, hyp, label in test_cases:
    w, S, D, I = wer(ref, hyp)
    print(f"  {ref:<28} {hyp:<32} {w:>5.1f}%  S={S} D={D} I={I}  [{label}]")

## 1. Word Error Rate (WER)

WER is the **standard metric** for evaluating ASR systems.
It measures how many word-level edits are needed to turn the hypothesis into the reference.

$$WER = \frac{S + D + I}{N} \times 100\%$$

| Symbol | Meaning | Example |
|--------|---------|---------|
| **S** | Substitution — wrong word | "dog" instead of "fox" |
| **D** | Deletion — missing word | skipped "the" |
| **I** | Insertion — extra word | added extra "um" |
| **N** | Total words in reference | denominator |

> WER can exceed 100% if there are many insertions!

### Benchmarks (LibriSpeech test-clean)

| Model | Year | WER |
|-------|------|-----|
| Human | — | ~5.8% |
| DeepSpeech 2 | 2015 | 5.33% |
| wav2vec 2.0 Large | 2020 | 1.8% |
| Whisper large-v3 | 2023 | 2.7% |
| Best published | 2024 | ~1.4% |

In [ ]:
# WER vs CER comparison
print("WER vs CER Comparison:")
print(f"  {'Reference':<22} {'Hypothesis':<22} {'WER':>6} {'CER':>6}  Notes")
print("  " + "-" * 72)

pairs = [
    ("hello world",       "helo world",          "1 char missing"),
    ("speech",            "speach",              "1 char substituted"),
    ("the cat",           "da cat",              "1 word wrong"),
    ("recognition",       "recgnition",          "1 char deleted in long word"),
    ("a big dog",         "a small dog",         "1 word sub, different lengths"),
]
for ref, hyp, note in pairs:
    w, *_ = wer(ref, hyp)
    c, *_ = cer(ref, hyp)
    print(f"  {ref:<22} {hyp:<22} {w:>5.1f}% {c:>5.1f}%  {note}")

print()
print("Key insight:")
print("  WER counts WORDS as units -> 'recognition' vs 'recgnition' = 1 word error = 100% WER")
print("  CER counts CHARS as units -> same example = 1 char error / 11 chars =  9% CER")

In [ ]:
## 2. Character Error Rate (CER)

CER uses the same edit distance formula but operates at the **character level**.

**When to use CER:**
- Languages without clear word boundaries: **Chinese, Thai, Japanese**
- Evaluating fine-grained TTS pronunciation quality
- When comparing systems that use different tokenization

**CER vs WER trade-off:**
- WER: one wrong word = 1 error regardless of word length
- CER: one wrong character = 1 error — more sensitive to small mistakes

## 3. WER at the IPA Level (Phoneme Error Rate)

The same edit distance formula works at **any granularity** — not just words and characters.

**Phoneme Error Rate (PER)** applies WER to IPA phoneme sequences:

```
Reference IPA:   /h ɛ l oʊ  w ɜː l d/
Hypothesis IPA:  /h ɛ l oʊ  w ɝ  l d/
                               ^
                           substitution: /ɜː/ -> /ɝ/

PER = 1 sub / 9 phonemes = 11.1%
```

**Why PER is useful:**
- Measures **pronunciation accuracy** more precisely than WER
- A model might produce the right word ("bird") but with a slightly wrong vowel (/bɜːd/ vs /bɝd/)
- PER catches fine-grained errors that WER and CER miss
- Used in **multilingual TTS evaluation** where word boundaries differ across languages

**Key insight:** WER, CER, and PER all use the same edit distance algorithm — only the **tokenization level** changes:

| Metric | Token unit | Use case |
|--------|-----------|---------|
| WER | Word | Standard ASR benchmark |
| CER | Character | Chinese/Thai/Japanese ASR |
| PER | IPA phoneme | Pronunciation accuracy, multilingual TTS |

In [ ]:
# PER — the same wer() function, just tokenize at the IPA phoneme level

def per(reference_ipa, hypothesis_ipa):
    # Input: list of IPA phoneme strings
    S, D, I, _ = edit_distance(reference_ipa, hypothesis_ipa)
    return (S + D + I) / max(len(reference_ipa), 1) * 100, S, D, I

# ARPAbet -> IPA mapping
arpabet_to_ipa = {
    "AA":"ɑ","AE":"æ","AH":"ʌ","AO":"ɔ","AW":"aʊ","AY":"aɪ",
    "EH":"ɛ","ER":"ɝ","EY":"eɪ","IH":"ɪ","IY":"iː","OW":"oʊ",
    "OY":"ɔɪ","UH":"ʊ","UW":"uː",
    "B":"b","CH":"tʃ","D":"d","DH":"ð","F":"f","G":"g","HH":"h",
    "JH":"dʒ","K":"k","L":"l","M":"m","N":"n","NG":"ŋ","P":"p",
    "R":"r","S":"s","SH":"ʃ","T":"t","TH":"θ","V":"v",
    "W":"w","Y":"j","Z":"z","ZH":"ʒ",
}

def arpabet_to_ipa_phones(phones):
    return [arpabet_to_ipa.get(p.rstrip("012"), "?") for p in phones]

# Example: "hello world" — correct vs slightly mispronounced
ref_arpabet = ["HH","EH0","L","OW1","W","ER1","L","D"]
hyp_arpabet = ["HH","AE0","L","OW1","W","ER1","L","D"]   # EH -> AE (wrong vowel)

ref_ipa = arpabet_to_ipa_phones(ref_arpabet)
hyp_ipa = arpabet_to_ipa_phones(hyp_arpabet)

w_score, *_ = wer("hello world", "hello world")   # WER: same words -> 0%
p_score, S, D, I = per(ref_ipa, hyp_ipa)          # PER: wrong vowel -> error

print("Transcript: 'hello world' (same words, wrong vowel in 'hello')")
print(f"  Reference IPA:   /{' '.join(ref_ipa)}/")
print(f"  Hypothesis IPA:  /{' '.join(hyp_ipa)}/")
print()
print(f"  WER = {w_score:.1f}%  (words match -> no error at word level)")
print(f"  PER = {p_score:.1f}%  (phoneme /ɛ/ was replaced by /æ/ -> caught!)")
print(f"        S={S} D={D} I={I} / {len(ref_ipa)} phonemes")
print()

# More examples
examples = [
    (["HH","EH0","L","OW1"], ["HH","EH0","L","OW1"],  "hello / hello (perfect)"),
    (["HH","EH0","L","OW1"], ["HH","AE0","L","OW1"],  "hello / hallo (wrong vowel)"),
    (["S","P","IY1","CH"],   ["S","P","IH1","CH"],     "speech / speech (tense->lax vowel)"),
    (["TH","R","UW1"],       ["T","R","UW1"],          "through / through (th->t, accent)"),
]

print("PER Examples:")
print(f"  {'Reference':<25} {'Hypothesis':<25} {'PER':>6}")
print("  " + "-" * 60)
for ref_p, hyp_p, label in examples:
    ref_i = arpabet_to_ipa_phones(ref_p)
    hyp_i = arpabet_to_ipa_phones(hyp_p)
    p, *_ = per(ref_i, hyp_i)
    print(f"  {label:<50} {p:>5.1f}%")

## Summary

| Metric | Token unit | Formula | Best for |
|--------|-----------|---------|---------|
| **WER** | Word | (S+D+I)/N | Standard English ASR benchmark |
| **CER** | Character | (S+D+I)/N | Chinese/Thai/Japanese ASR |
| **PER** | IPA phoneme | (S+D+I)/N | Pronunciation accuracy, multilingual TTS |

All three metrics use the **same edit distance algorithm** — the only difference is what counts as a "token".

**Where these metrics appear in the course:**
- **03-STT**: CTC and attention-based models are trained to minimize WER/CER
- **02-TTS**: run ASR on synthesized speech -> compute WER against original text
- **04-VoiceCloning**: PER helps evaluate whether the cloned voice pronounces correctly